<a href="https://colab.research.google.com/github/MithunSrinivas28/wafer-defect-ai/blob/final/Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

BLOCK 1 — IMPORTS

In [436]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [437]:
import tensorflow as tf
import numpy as np
import os
import shutil
import csv
from datetime import datetime


In [438]:
SELF_LEARN_DIR = "/content/drive/MyDrive/Datasets/Self-learning"

os.makedirs(SELF_LEARN_DIR, exist_ok=True)


In [439]:
OFFLINE_SYNC_DIR = BASE_DIR + "/Offline_Storage/Pending_Sync"
os.makedirs(OFFLINE_SYNC_DIR, exist_ok=True)


In [440]:
SYNCED_DIR = BASE_DIR + "/Offline_Storage/Synced"

os.makedirs(SYNCED_DIR, exist_ok=True)


In [441]:
# ===== BASE PATH =====

BASE_DIR = "/content/drive/MyDrive/Wafer_Pipeline"

MODEL_PATH = BASE_DIR + "/wafer_xai2_model.keras"


In [442]:
# ===== CREATE MAIN FOLDERS =====

folders = [
    BASE_DIR,
    BASE_DIR + "/Input_Images",
    BASE_DIR + "/Results",
    BASE_DIR + "/Datasets/Self-learning",
    BASE_DIR + "/Offline_Storage/Pending_Sync"
]

for f in folders:
    os.makedirs(f, exist_ok=True)

print("Folder structure created")


Folder structure created


In [443]:
# ===== LOAD MODEL =====

model = tf.keras.models.load_model(MODEL_PATH)

print("Model loaded successfully")


Model loaded successfully


In [444]:
# ===== CLASS NAMES (MUST MATCH TRAINING ORDER) =====

CLASS_NAMES = [
    'bridge','clean','cmp','crack',
    'ler','open','others','vias'
]

print("Classes:", CLASS_NAMES)


Classes: ['bridge', 'clean', 'cmp', 'crack', 'ler', 'open', 'others', 'vias']


#  IMAGE PREPROCESSING

In [445]:
# ===== IMAGE PREPROCESSING =====

IMG_SIZE = 224


def preprocess_image(img_path):

    img = tf.keras.utils.load_img(
        img_path,
        color_mode="grayscale",
        target_size=(IMG_SIZE, IMG_SIZE)
    )

    img = tf.keras.utils.img_to_array(img)

    # Normalize
    img = img / 255.0

    # Gray → RGB
    img = tf.repeat(img, 3, axis=-1)

    # Add batch dimension
    img = tf.expand_dims(img, axis=0)

    return img


In [446]:
# List files in Input_Images folder

input_files = os.listdir(BASE_DIR + "/Input_Images")

print("Images found:", input_files)


Images found: ['crack.png', 'cmp.png', 'open.png', 'test1.png', 'vias.png', 'otherss.png', 'bri.png', 'vias2.png', 'test2.png', '1.png', 'wall2.png', 'snake1.png', 'wall1.png']


In [447]:
# ===== SINGLE IMAGE PREDICTION =====

def predict_image(img_path):

    img = preprocess_image(img_path)

    preds = model.predict(img, verbose=0)[0]

    class_id = np.argmax(preds)

    confidence = float(np.max(preds))

    label = CLASS_NAMES[class_id]

    return label, confidence


In [448]:
# ===== TEST ON ONE IMAGE =====

test_image = BASE_DIR + "/Input_Images/open.png"

label, conf = predict_image(test_image)

print("Prediction :", label)
print("Confidence :", round(conf*100, 2), "%")


Prediction : open
Confidence : 54.81 %


# **✅ STEP 3 — Low-Confidence → Self-Learning Folder**

In [449]:
import shutil
from datetime import datetime


In [450]:
# ===== CONFIDENCE THRESHOLD =====

CONF_THRESHOLD = 0.40   # 30%


In [451]:
def inspect_image(img_path):

    img_name = os.path.basename(img_path)

    img = preprocess_image(img_path)

    preds = model.predict(img, verbose=0)[0]

    class_id = np.argmax(preds)

    confidence = float(np.max(preds))

    label = CLASS_NAMES[class_id]


    # ---- CHECK CONFIDENCE ----
    if confidence < CONF_THRESHOLD:

        status = "LOW_CONFIDENCE"

        # Copy to self-learning folder
        shutil.copy(
            img_path,
            os.path.join(SELF_LEARN_DIR, img_name)
        )

        print("⚠ Sent to Self-learning folder")

    else:
        status = "OK"


    return label, confidence, status


In [452]:
def run_inspection(image_path):

    label, conf, status = inspect_image(image_path)

    print("Image :", os.path.basename(image_path))
    print("Prediction :", label)
    print("Confidence :", round(conf*100,2), "%")
    print("Status :", status)


In [453]:
test_image = BASE_DIR + "/Input_Images/open.png"

run_inspection(test_image)


Image : open.png
Prediction : open
Confidence : 54.81 %
Status : OK


# STEP 4.1

In [454]:
import csv
from datetime import datetime


In [455]:
CSV_PATH = BASE_DIR + "/Results/predictions.csv"


In [456]:
def init_csv():

    if not os.path.exists(CSV_PATH):

        with open(CSV_PATH, "w", newline="") as f:

            writer = csv.writer(f)

            writer.writerow([
                "Timestamp",
                "Image_Name",
                "Prediction",
                "Confidence",
                "Status"
            ])

        print("CSV file created")


init_csv()


In [457]:
def inspect_image(img_path):

    img_name = os.path.basename(img_path)

    img = preprocess_image(img_path)

    preds = model.predict(img, verbose=0)[0]

    class_id = np.argmax(preds)

    confidence = float(np.max(preds))

    label = CLASS_NAMES[class_id]


    # ---- CHECK CONFIDENCE ----
    if confidence < CONF_THRESHOLD:

        status = "LOW_CONFIDENCE"

        shutil.copy(
            img_path,
            os.path.join(SELF_LEARN_DIR, img_name)
        )

        print("⚠ Sent to Self-learning folder")

    else:
        status = "OK"


    # ---- LOG TO CSV (OFFLINE HISTORY) ----
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")


    with open(CSV_PATH, "a", newline="") as f:

        writer = csv.writer(f)

        writer.writerow([
            timestamp,
            img_name,
            label,
            round(confidence,4),
            status
        ])


    return label, confidence, status


In [458]:
run_inspection(BASE_DIR + "/Input_Images/open.png")


Image : open.png
Prediction : open
Confidence : 54.81 %
Status : OK


In [459]:
import pandas as pd

pd.read_csv(CSV_PATH)


,Timestamp,Image_Name,Prediction,Confidence,Status
0,2026-02-06 11:59:46,crack.png,crack,0.7178,OK
1,2026-02-06 12:00:29,crack.png,crack,0.7178,OK
2,2026-02-06 12:01:00,open.png,open,0.5481,OK
3,2026-02-06 12:10:11,crack.png,crack,0.7178,OK
4,2026-02-06 12:10:11,cmp.png,cmp,0.4624,OK
5,2026-02-06 12:10:12,open.png,open,0.5481,OK
6,2026-02-06 12:10:12,test1.png,ler,0.3806,LOW_CONFIDENCE
7,2026-02-06 12:10:12,vias.png,vias,0.5703,OK
8,2026-02-06 12:10:12,otherss.png,others,0.8316,OK
9,2026-02-06 12:10:12,bri.png,bridge,0.5373,OK


# STEP 5 — Batch Folder Scanner

In [460]:
def run_batch_inspection():

    files = os.listdir(INPUT_DIR)

    if len(files) == 0:
        print("No images found in Input_Images folder")
        return


    print("Starting Batch Inspection...")
    print("Total Images:", len(files))
    print("-" * 40)


    defect_count = {}
    low_conf_count = 0


    for file in files:

        img_path = os.path.join(INPUT_DIR, file)

        # Skip non-image files
        if not file.lower().endswith((".png", ".jpg", ".jpeg")):
            continue


        label, conf, status = inspect_image(img_path)


        # Count defects
        defect_count[label] = defect_count.get(label, 0) + 1


        if status == "LOW_CONFIDENCE":
            low_conf_count += 1


        print(f"{file:20s} → {label:8s} ({conf*100:.1f}%) {status}")


    # ---- SUMMARY ----
    print("\nBatch Summary")
    print("-" * 40)

    total = sum(defect_count.values())

    for k, v in defect_count.items():
        print(f"{k:8s} : {v}")

    print("\nLow Confidence :", low_conf_count)

    yield_percent = (defect_count.get("clean", 0) / total) * 100

    print("Yield :", round(yield_percent, 2), "%")

    if yield_percent < 70:
        print("Batch Status : FAIL")
    else:
        print("Batch Status : PASS")


In [461]:
INPUT_DIR = BASE_DIR + "/Input_Images"


In [462]:
run_batch_inspection()


Starting Batch Inspection...
Total Images: 13
----------------------------------------
crack.png            → crack    (71.8%) OK
cmp.png              → cmp      (46.2%) OK
open.png             → open     (54.8%) OK
⚠ Sent to Self-learning folder
test1.png            → ler      (38.1%) LOW_CONFIDENCE
vias.png             → vias     (57.0%) OK
otherss.png          → others   (83.2%) OK
bri.png              → bridge   (53.7%) OK
vias2.png            → vias     (54.5%) OK
test2.png            → vias     (47.4%) OK
1.png                → open     (45.2%) OK
⚠ Sent to Self-learning folder
wall2.png            → cmp      (33.2%) LOW_CONFIDENCE
snake1.png           → others   (42.7%) OK
wall1.png            → others   (44.3%) OK

Batch Summary
----------------------------------------
crack    : 1
cmp      : 2
open     : 2
ler      : 1
vias     : 3
others   : 3
bridge   : 1

Low Confidence : 2
Yield : 0.0 %
Batch Status : FAIL


# STEP 6 — Offline Sync System

In [463]:
SYNCED_DIR = BASE_DIR + "/Offline_Storage/Synced"

os.makedirs(SYNCED_DIR, exist_ok=True)


In [464]:
def sync_data():

    pending_files = os.listdir(OFFLINE_SYNC_DIR)

    if len(pending_files) == 0:
        print("Nothing to sync")
        return


    print("Starting Sync...")
    print("-" * 30)


    synced_count = 0


    for file in pending_files:

        src = os.path.join(OFFLINE_SYNC_DIR, file)

        dst = os.path.join(SYNCED_DIR, file)


        # Avoid overwriting
        if os.path.exists(dst):
            continue


        shutil.move(src, dst)

        synced_count += 1


        print(f"Synced: {file}")


    print("-" * 30)
    print("Sync Complete")
    print("Files synced:", synced_count)


In [465]:
run_batch_inspection()


Starting Batch Inspection...
Total Images: 13
----------------------------------------
crack.png            → crack    (71.8%) OK
cmp.png              → cmp      (46.2%) OK
open.png             → open     (54.8%) OK
⚠ Sent to Self-learning folder
test1.png            → ler      (38.1%) LOW_CONFIDENCE
vias.png             → vias     (57.0%) OK
otherss.png          → others   (83.2%) OK
bri.png              → bridge   (53.7%) OK
vias2.png            → vias     (54.5%) OK
test2.png            → vias     (47.4%) OK
1.png                → open     (45.2%) OK
⚠ Sent to Self-learning folder
wall2.png            → cmp      (33.2%) LOW_CONFIDENCE
snake1.png           → others   (42.7%) OK
wall1.png            → others   (44.3%) OK

Batch Summary
----------------------------------------
crack    : 1
cmp      : 2
open     : 2
ler      : 1
vias     : 3
others   : 3
bridge   : 1

Low Confidence : 2
Yield : 0.0 %
Batch Status : FAIL


In [466]:
sync_data()


Nothing to sync
